In [11]:
using Gmsh: gmsh

# Dimensions
L = 1000.0   # x-direction (length)
W = 250.0    # y-direction (width)
H = 200.0    # z-direction (height)
h = 10   # mesh size

gmsh.initialize()
gmsh.model.add("Cantilever_Q5")

# --- Create the 8 corner points of the box ---
p1 = gmsh.model.geo.addPoint(0, 0, 0, h)
p2 = gmsh.model.geo.addPoint(L, 0, 0, h)
p3 = gmsh.model.geo.addPoint(L, W, 0, h)
p4 = gmsh.model.geo.addPoint(0, W, 0, h)

p5 = gmsh.model.geo.addPoint(0, 0, H, h)
p6 = gmsh.model.geo.addPoint(L, 0, H, h)
p7 = gmsh.model.geo.addPoint(L, W, H, h)
p8 = gmsh.model.geo.addPoint(0, W, H, h)

# --- Create lines for bottom face ---
l1 = gmsh.model.geo.addLine(p1, p2)
l2 = gmsh.model.geo.addLine(p2, p3)
l3 = gmsh.model.geo.addLine(p3, p4)
l4 = gmsh.model.geo.addLine(p4, p1)

# --- Create lines for top face ---
l5 = gmsh.model.geo.addLine(p5, p6)
l6 = gmsh.model.geo.addLine(p6, p7)
l7 = gmsh.model.geo.addLine(p7, p8)
l8 = gmsh.model.geo.addLine(p8, p5)

# --- Side edges ---
l9  = gmsh.model.geo.addLine(p1, p5)
l10 = gmsh.model.geo.addLine(p2, p6)
l11 = gmsh.model.geo.addLine(p3, p7)
l12 = gmsh.model.geo.addLine(p4, p8)

# ---- Create surfaces (6 faces of the box) ----

# Bottom rectangle
cl1 = gmsh.model.geo.addCurveLoop([l1, l2, l3, l4])
s1  = gmsh.model.geo.addPlaneSurface([cl1])

# Top rectangle
cl2 = gmsh.model.geo.addCurveLoop([l5, l6, l7, l8])
s2  = gmsh.model.geo.addPlaneSurface([cl2])

# Side faces
cl3 = gmsh.model.geo.addCurveLoop([l1, l10, -l5, -l9])
s3  = gmsh.model.geo.addPlaneSurface([cl3])

cl4 = gmsh.model.geo.addCurveLoop([l2, l11, -l6, -l10])
s4  = gmsh.model.geo.addPlaneSurface([cl4])

cl5 = gmsh.model.geo.addCurveLoop([l3, l12, -l7, -l11])
s5  = gmsh.model.geo.addPlaneSurface([cl5])

cl6 = gmsh.model.geo.addCurveLoop([l4, l9, -l8, -l12])
s6  = gmsh.model.geo.addPlaneSurface([cl6])

# ---- Create 3D volume ----
sl = gmsh.model.geo.addSurfaceLoop([s1, s2, s3, s4, s5, s6])
vol = gmsh.model.geo.addVolume([sl])

gmsh.model.geo.synchronize()

# ---- Physical groups ----

# 3D domain
gmsh.model.addPhysicalGroup(3, [vol], 10, "Domain")

# Fixed End (x = 0 → face s6)
gmsh.model.addPhysicalGroup(2, [s6], 2)
gmsh.model.setPhysicalName(2, 2, "FixedEnd")

# Load End (x = L → face s4)
gmsh.model.addPhysicalGroup(2, [s4], 3)
gmsh.model.setPhysicalName(2, 3, "LoadFace")

# ---- Mesh ----
gmsh.model.mesh.generate(3)

# Export mesh
gmsh.write("Cantilever_Q5.msh")

# View mesh
gmsh.fltk.run()
gmsh.finalize()


Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 10%] Meshing curve 2 (Line)
Info    : [ 20%] Meshing curve 3 (Line)
Info    : [ 30%] Meshing curve 4 (Line)
Info    : [ 40%] Meshing curve 5 (Line)
Info    : [ 50%] Meshing curve 6 (Line)
Info    : [ 60%] Meshing curve 7 (Line)
Info    : [ 60%] Meshing curve 8 (Line)
Info    : [ 70%] Meshing curve 9 (Line)
Info    : [ 80%] Meshing curve 10 (Line)
Info    : [ 90%] Meshing curve 11 (Line)
Info    : [100%] Meshing curve 12 (Line)
Info    : Done meshing 1D (Wall 0.00342131s, CPU 0s)
Info    : Meshing 2D...
Info    : [  0%] Meshing surface 1 (Plane, Frontal-Delaunay)
Info    : [ 20%] Meshing surface 2 (Plane, Frontal-Delaunay)
Info    : [ 40%] Meshing surface 3 (Plane, Frontal-Delaunay)
Info    : [ 60%] Meshing surface 4 (Plane, Frontal-Delaunay)
Info    : [ 70%] Meshing surface 5 (Plane, Frontal-Delaunay)
Info    : [ 90%] Meshing surface 6 (Plane, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 0.339201s, CPU 0.2

In [2]:
using Gridap
using GridapGmsh

In [3]:
# 1. Read the Mesh
model = GmshDiscreteModel("Cantilever_Q5.msh")


Info    : Reading 'Cantilever_Q5.msh'...
Info    : 27 entities
Info    : 42231 nodes
Info    : 229015 elements
Info    : Done reading 'Cantilever_Q5.msh'                                                                       


UnstructuredDiscreteModel()

In [4]:
# 2. Material Parameters (Steel)
const E = 25000.0  # MPa
const ν = 0.2       # Poisson's ratio

0.2

In [5]:
# Lamé parameters for Plane Stress

g = VectorValue(0.0,0.0,-1000.0/(250*200)) ## force

const λ = (E*ν)/((1+ν)*(1-2*ν))
const μ = E/(2*(1+ν))
σ(ε) = λ*tr(ε)*one(ε) + 2*μ*ε

σ (generic function with 1 method)

In [6]:
# 3. Define FE Spaces
order = 1

reffe = ReferenceFE(lagrangian,VectorValue{3,Float64},order)
V0 = TestFESpace(model,reffe;
    conformity=:H1,
    dirichlet_tags=["FixedEnd"],
    dirichlet_masks=[(true,true,true)])
  
g1 = VectorValue(0.0,0.0,0.0)
U = TrialFESpace(V0,[g1])

TrialFESpace()

In [7]:
# 4. Numerical Integration
degree = 2*order
Ω = Triangulation(model)
dΩ = Measure(Ω,degree)
Γ_load  = BoundaryTriangulation(model, tags = "LoadFace")
dΓ_N = Measure(Γ_load,degree)

GenericMeasure()

In [8]:
# Weak from

# Internal work (Stiffness component)
a(u,v) = ∫( ε(v) ⊙ (σ∘ε(u)) )*dΩ 

# External Work
l(v) = ∫(v⋅g)*dΓ_N

l (generic function with 1 method)

In [9]:
# Solve
op = AffineFEOperator(a,l,U,V0)
uh = solve(op)

SingleFieldFEFunction():
 num_cells: 226661
 DomainStyle: ReferenceDomain()
 Triangulation: BodyFittedTriangulation()
 Triangulation id: 16655051525491227965

In [10]:
# 7. Post-Process
writevtk(Ω,"results_Q5",
    cellfields=[
        "Displacement"=>uh,
        "Strain"=>ε(uh),
        "Stress"=>σ∘ε(uh)]
        )

(["results_Q5.vtu"],)